# 확장 노면온도 모델 제출 검토

현재 v3의 고정 학습 계획, 실제 추론, 지역 제외 개발 지표와 제출 CSV를 검산합니다. 기본 실행은 재학습하지 않습니다. 보호 3곳의 정답은 포함하지 않습니다.

In [ ]:
from pathlib import Path
import json, sys
import joblib, numpy as np, pandas as pd
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'extended_training_plan.json').exists())
sys.path.insert(0, str(ROOT))
from src.data_source import sha256_file as sha
from src.extended_runtime import infer
from src.submission_io import read_weather
from src.offline_runtime import deny_network
from src.temperature_transfer import regression_report
from src.runtime_metrics import runtime_daily
read = lambda p: json.loads((ROOT/p).read_text('utf-8'))
plan = read('extended_training_plan.json')
assert not plan['protected_outcomes_used'] and not plan['new_independent_assessment']
plan['periods']

## 1. 포함 자료와 최종 모델

허용 23곳의 두 겨울 원시 794,545행에서 과거 특징을 먼저 계산합니다. 입력·온도 정답이 모두 있는 최종 학습 쌍은 793,149행입니다. 온도와 결빙 기준 모델의 학습 입력을 구분합니다.

In [ ]:
for role, entry in plan['training_sources'].items():
    assert sha(ROOT/entry['file']) == entry['sha256']
    identities = pd.read_parquet(ROOT/entry['file'], columns=['station','time'])
    assert set(identities.station) == set(plan['allowed_stations'])
    assert not set(identities.station) & {'12410','33411','23402'}
receipts = read('model/refit_receipt.json')
for entry in receipts['models']:
    assert sha(ROOT/'model'/(entry['name']+'.joblib')) == entry['sha256']
pd.DataFrame(receipts['models'])[['name','role','rows']]

## 2. 과거 이력과 실제 추론

216행 입력의 첫 72행은 이력 구성용입니다. 예측 후 하루 144행만 선택합니다. 학습 범위 내 예제이므로 독립 성능평가가 아닙니다.

In [ ]:
models = {r:[joblib.load(ROOT/'model'/(s['name']+'.joblib')) for s in plan['models'] if s['role']==r] for r in ['temperature','icing_reference']}
with deny_network() as attempts:
    actual = infer(read_weather(ROOT/'data/weather_example.csv'), models['temperature'], models['icing_reference'], plan['temperature_feature_kind'], plan['temperature_weights'])
actual = actual.loc[actual.time.ge('2024-12-15') & actual.time.lt('2024-12-16')].reset_index(drop=True)
expected = pd.read_csv(ROOT/'data/example_predictions.csv', float_precision='round_trip', dtype={'station':str,'predicted_ice':'Int64'}, parse_dates=['time'])
pd.testing.assert_frame_equal(actual, expected, check_exact=True)
assert not attempts and len(actual)==144
actual[['time','estimated_road_temp_c','icing_reference_temperature_c','predicted_ice']].head()

## 3. 지역을 제외한 개발 지표

평가 지역의 두 겨울 전체를 학습에서 제외한 예측입니다. 여러 후보를 같은 9곳에서 비교했으므로 새 독립 평가로 표현하지 않습니다. 현재 온도와 유지된 결빙 판정을 별도로 검산합니다.

In [ ]:
frame = pd.read_parquet(ROOT/'research/extended/audited_predictions.parquet')
audit = read('research/extended/audit.json')
assert sha(ROOT/'research/extended/audited_predictions.parquet') == audit['runtime_prediction_sha256']
temperature = regression_report(frame, frame.temperature_both)
ice, stations, days = runtime_daily(frame, frame.physical_both)
np.testing.assert_allclose(temperature['pooled']['mae'], audit['temperature']['regression']['pooled']['mae'], atol=1e-12, rtol=0)
np.testing.assert_allclose(ice['f1_fixed_population'], audit['physical']['both']['all']['f1_fixed_population'], atol=1e-12, rtol=0)
print('온도 MAE:',temperature['pooled']['mae'])
print('일별 지점 F1:',ice['f1_fixed_population'])
pd.DataFrame(temperature['stations']).T

## 4. 제출 CSV와 보호 예측

138개 공유 기상 그룹을 위치 인덱스로 연결합니다. 구간 수는 독립 검증 표본 수가 아닙니다. 보호 예측에는 관측 정답이 없고 입력 부족 18행은 NA입니다.

In [ ]:
exports = read('exports/manifest.json')
assert exports['development_scope']=='extended_two_winter'
for table in exports['tables']:
    assert sha(ROOT/'exports'/table['file']) == table['sha256']
protected = pd.read_csv(ROOT/'exports/protected_predictions_10min.csv', dtype={'station':str})
assert len(protected)==103680 and protected.predicted_ice.isna().sum()==18
assert not {'road_temp','ice','observed_ice','actual_ice'} & set(protected)
pd.DataFrame(exports['tables'])[['file','rows','bytes']]

## 재학습

`python -X utf8 code/train.py --output validation/my_refit`은 포함 자료에서 고정 모델을 새 폴더에 적합합니다. 실제 독립 재학습의 모델별 해시 일치는 `validation/extended_delivery.json`에 기록합니다. 전체 탐색 재실행은 원 비공개 작업 공간의 자료가 필요합니다. 현재 보고서는 `reports/current/current_report.pdf`, 실행 화면은 `demo/index.html`입니다.